<a href="https://colab.research.google.com/github/iespinozahDM/UBA-DM/blob/main/Colab_Base_para_el_Trabajo_Pr%C3%A1ctico_(versi%C3%B3n_0).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Base para el Trabajo Práctico (versión 0)
Programa de creación de entregas.

In [1]:
import pandas as pd
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 0. Lectura de datos

In [ ]:
# TODO: Cambiar para que apunte al directorio correcto
DIR = "/content/drive/MyDrive/DM/2026/datos"

In [ ]:
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")

df_ent.pd.read_sql("SELECT * FROM propiedades", engine)
df_ent.head(3)

In [ ]:
# cantidad de filas y columnas
df_ent.shape

In [ ]:
df_ent.columns

## 1. Entender los datos (AID)

In [ ]:
df_ent["source"].hist(figsize=(15,5));

## 2. Limpiar y transformar los datos (DM)

In [ ]:
# Selección de datos. Solo a fines demostrativos.
# TODO: CAMBIAR!
df_ent = df_ent.loc[(df_ent["location_1"] == "Santa Fe") & (df_ent["operation_type"] == 'alquiler') & (df_ent["property_type"] == "departamento")]
df_ent.shape

In [ ]:
# La creación de modelos requiere que no haya valores perdidos
# por eso llenamos todo con 0 a lo bestia
# TODO: mejorar la imputación de valores perdidos
df_ent = df_ent.fillna(0)

## 3. Entrenamiento del modelos (AA) - ⛔⛔⛔ NO TOCAR ⛔⛔⛔

In [ ]:
# La creación de modelos requiere que todo el dataframe sea numérico
# Me quedo con las columnas numéricas solamente
# TODO: traducir las columnas con datos no numéricos a numéricos para que mejoren los modelos
df_ent = df_ent.select_dtypes('number')

X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

In [ ]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Definimos el valor de los hiperparámetros a usar por el modelo
n_estimators = 50
max_depth = 5

### NO CAMBIAR RandomForestRegressor por otro modelo
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

# Entrenamos el modelo
_ = reg.fit(X_train, y_train)

# Cálculo del error en entrenamiento (train)
y_pred = reg.predict(X_train)
score_train = sk.metrics.root_mean_squared_error(y_train, y_pred)

# Cálculo del error en prueba (test)
y_pred = reg.predict(X_test)
score_test  = sk.metrics.root_mean_squared_error(y_test,  y_pred)

print(f"{n_estimators=} -- {max_depth=} --> {score_train=:.2f} - {score_test=:.2f}")

## 4. Solución para subir Kaggle

In [ ]:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")
df_ap.head(2)

In [ ]:
df_ap.shape

In [ ]:
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

# Entrenamos el modelo con todos los datos de entrenamiento.csv
reg.fit(X, y)

In [ ]:
# Hacemos en df_ap la misma limpieza que en df_ent
df_ap = df_ap.fillna(0)

df_ap = df_ap.select_dtypes('number')

X_ap = df_ap[X.columns]

# Predecimos los precios del dataset a predecir
y_pred_ap = reg.predict(X_ap)
y_pred_ap

In [ ]:
# Lleno el precio de df_ap con las predicciones
df_ap["price"] = y_pred_ap

# Grabo el df_ap en un archivo csv para subir a Kaggle
df_ap["price"].to_csv("solucion.csv")